# 04 - DEPRECATED

**This notebook is no longer used.** LLM batch ingestion is now handled by
`scripts/migrate/ingest_llm_from_db.py` (per the schema migration to
`ai_narratives_original`). The script ingests all three models in one pass.

To repopulate LLM data after a new batch run lands:

```bash
uv run python scripts/migrate/ingest_llm_from_db.py
```

Refer to `docs/rebuttal/SCHEMA_MIGRATION.md` for the full ingestion pipeline.

Original notebook content preserved below for reference only — do not re-run.


# 04 - Batch Repetitions Data Extraction

This notebook extracts LLM batch results from a specified experiment group and loads them into the ACM 202512 schema with the appropriate `run_number`.

**Strategy:** Since runs 2-10 reuse the same `narrative_id`s from run 1 (groups 35-36), we use a direct `narrative_id -> participant_id` mapping from the existing run 1 data in the ACM schema. No hash recomputation needed.

**Usage:**
1. Set `EXPERIMENTS_GROUP_ID` to the experiment group from the batch run
2. Set `RUN_NUMBER` to the repetition number (2, 3, ..., 10)
3. Run all cells
4. Verify the correct number of records were inserted

```bash
uv run python scripts/run_batch_evaluation.py \
--from-groups 35,36 \
--run-number 3 \
--model gpt-5 \
--description "Batch Run 3 - ACM Publication" \
--yes
```

## 1. Setup & Configuration

In [ ]:
# ============================================================================
# PARAMETERS - SET THESE BEFORE RUNNING
# ============================================================================
# (38,2), (39,3)
EXPERIMENTS_GROUP_ID = 39  # e.g., 37, 38, etc.
RUN_NUMBER = 3  # e.g., 2, 3, ..., 10

# Validate parameters
assert EXPERIMENTS_GROUP_ID is not None, "Please set EXPERIMENTS_GROUP_ID"
assert RUN_NUMBER is not None, "Please set RUN_NUMBER"
assert RUN_NUMBER >= 2 and RUN_NUMBER <= 10, "RUN_NUMBER should be between 2 and 10"

print(f"Configuration:")
print(f"  Experiment Group ID: {EXPERIMENTS_GROUP_ID}")
print(f"  Run Number: {RUN_NUMBER}")

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from sqlmodel import Session

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / 'src'))

# Import database manager
from pain_narratives.core.database import DatabaseManager

# Initialize database manager
db = DatabaseManager()

print(f"Connected to database")

In [ ]:
# Schema names
SOURCE_SCHEMA = 'pain_narratives_app'
TARGET_SCHEMA = 'pain_narratives_acm_202512'

## 2. Load Participant Mapping (Direct from Run 1)

In [ ]:
# Load the direct narrative_id -> participant_id mapping from run 1 data
# Since runs 2-10 reuse the same narrative_ids, we can get the mapping directly
# by joining source experiments (groups 35-36) with ACM schema results

query_mapping = f"""
SELECT DISTINCT
    el.narrative_id,
    de.participant_id
FROM {SOURCE_SCHEMA}.experiments_list el
JOIN {TARGET_SCHEMA}.llm_dimension_evaluation de 
    ON el.experiment_id = de.experiment_id
WHERE el.experiments_group_id IN (35, 36)
ORDER BY el.narrative_id
"""

with Session(db.engine) as session:
    mapping_df = pd.read_sql(text(query_mapping), session.connection())

print(f"Loaded {len(mapping_df)} narrative_id -> participant_id mappings from run 1")

# Create mapping dictionary
narrative_to_participant = dict(zip(mapping_df['narrative_id'], mapping_df['participant_id']))

# Verify we have all 152 mappings
if len(narrative_to_participant) != 152:
    print(f"*** WARNING: Expected 152 mappings, got {len(narrative_to_participant)} ***")

In [ ]:
# Show sample mappings
print("Sample narrative_id -> participant_id mappings:")
for nid, pid in list(narrative_to_participant.items())[:5]:
    print(f"  narrative_id {nid} -> participant_id {pid}")

## 3. Load Experiment Data

In [ ]:
# Query experiments from the specified group
query_experiments = f"""
SELECT 
    el.experiment_id,
    el.experiments_group_id,
    el.narrative_id,
    el.model,
    el.succeeded
FROM {SOURCE_SCHEMA}.experiments_list el
WHERE el.experiments_group_id = {EXPERIMENTS_GROUP_ID}
  AND el.succeeded = true
"""

with Session(db.engine) as session:
    experiments_df = pd.read_sql(text(query_experiments), session.connection())

print(f"Loaded {len(experiments_df)} experiments from group {EXPERIMENTS_GROUP_ID}")

if len(experiments_df) != 152:
    print(f"\n*** WARNING: Expected 152 experiments, got {len(experiments_df)} ***")
    print("Please verify all narratives were processed successfully before continuing.")

In [ ]:
# Map narrative_id to participant_id using the direct mapping from run 1
experiments_df['participant_id'] = experiments_df['narrative_id'].map(narrative_to_participant)

# Check mapping success
matched = experiments_df['participant_id'].notna().sum()
unmatched = experiments_df['participant_id'].isna().sum()
print(f"Mapping results:")
print(f"  Matched: {matched}")
print(f"  Unmatched: {unmatched}")

if unmatched > 0:
    print("\n*** WARNING: Some narratives could not be matched! ***")
    print("These narrative_ids are not in run 1 data:")
    print(experiments_df[experiments_df['participant_id'].isna()][['experiment_id', 'narrative_id']])

In [ ]:
# Create experiment_id -> participant_id mapping for later use
# Also create experiment_id -> model mapping
exp_to_participant = dict(zip(experiments_df['experiment_id'], experiments_df['participant_id'].astype(int)))
exp_to_model = dict(zip(experiments_df['experiment_id'], experiments_df['model']))

print(f"Models used: {experiments_df['model'].unique()}")

## 4. Extract Dimension Evaluations

In [ ]:
# Query dimension evaluation results
query_dimensions = f"""
SELECT 
    er.experiment_id,
    er.experiments_group_id,
    er.result_json
FROM {SOURCE_SCHEMA}.evaluation_results er
WHERE er.experiments_group_id = {EXPERIMENTS_GROUP_ID}
  AND er.result_type = 'dimensions'
"""

with Session(db.engine) as session:
    dimensions_raw_df = pd.read_sql(text(query_dimensions), session.connection())

print(f"Loaded {len(dimensions_raw_df)} dimension evaluation records")

In [ ]:
# Parse dimension results
def parse_dimension_result(result_json):
    """Extract severidad and discapacidad from result JSON."""
    if not result_json:
        return None, None, None, None
    
    result = result_json if isinstance(result_json, dict) else json.loads(result_json)
    
    # Try different key names
    severidad_data = result.get('severidad_del_dolor') or result.get('Severidad del dolor') or result.get('severidad') or {}
    discapacidad_data = result.get('discapacidad') or result.get('Discapacidad') or {}
    
    severidad_score = severidad_data.get('puntuacion') if isinstance(severidad_data, dict) else None
    severidad_exp = severidad_data.get('explicacion') if isinstance(severidad_data, dict) else None
    discapacidad_score = discapacidad_data.get('puntuacion') if isinstance(discapacidad_data, dict) else None
    discapacidad_exp = discapacidad_data.get('explicacion') if isinstance(discapacidad_data, dict) else None
    
    return severidad_score, severidad_exp, discapacidad_score, discapacidad_exp

# Extract dimension data
dimension_records = []
for _, row in dimensions_raw_df.iterrows():
    sev_score, sev_exp, dis_score, dis_exp = parse_dimension_result(row['result_json'])
    dimension_records.append({
        'experiment_id': row['experiment_id'],
        'experiments_group_id': row['experiments_group_id'],
        'severidad_score': sev_score,
        'severidad_explicacion': sev_exp,
        'discapacidad_score': dis_score,
        'discapacidad_explicacion': dis_exp
    })

df_llm_dimensions = pd.DataFrame(dimension_records)
df_llm_dimensions['participant_id'] = df_llm_dimensions['experiment_id'].map(exp_to_participant)
df_llm_dimensions['model'] = df_llm_dimensions['experiment_id'].map(exp_to_model)
df_llm_dimensions['run_number'] = RUN_NUMBER

# Filter to valid records
df_llm_dimensions = df_llm_dimensions[df_llm_dimensions['participant_id'].notna()].copy()

# Reorder columns
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'severidad_score', 'severidad_explicacion', 'discapacidad_score', 'discapacidad_explicacion']
df_llm_dimensions = df_llm_dimensions[cols_order]

print(f"\nExtracted dimension scores for {len(df_llm_dimensions)} records")
print(df_llm_dimensions[['participant_id', 'severidad_score', 'discapacidad_score', 'model']].head())

## 5. Extract PCS Results

In [ ]:
# Query PCS questionnaire results
query_pcs = f"""
SELECT 
    er.experiment_id,
    er.experiments_group_id,
    er.result_json
FROM {SOURCE_SCHEMA}.evaluation_results er
WHERE er.experiments_group_id = {EXPERIMENTS_GROUP_ID}
  AND er.result_type = 'PCS'
"""

with Session(db.engine) as session:
    pcs_raw_df = pd.read_sql(text(query_pcs), session.connection())

print(f"Loaded {len(pcs_raw_df)} PCS records")

In [ ]:
def parse_pcs_result(result_json):
    """Extract PCS data from result JSON.
    
    Handles two formats:
    - Old format (run 1): scores at top level, subscales as direct keys
    - New format (runs 2+): data nested in raw_data, scores with numeric keys, subscales in dict
    """
    if not result_json:
        return {}
    
    result = result_json if isinstance(result_json, dict) else json.loads(result_json)
    
    # Check if this is the new format (has raw_data key)
    if 'raw_data' in result:
        raw_data = result['raw_data']
        scores_dict = raw_data.get('scores', {})
        persona = raw_data.get('persona', {})
        model_reasoning = raw_data.get('model_reasoning')
        
        # Extract subscales from new format
        subscales = result.get('subscales', {})
        pcs_rumination = subscales.get('rumination')
        pcs_magnification = subscales.get('magnification')
        pcs_helplessness = subscales.get('helplessness')
        
        # Individual items - new format uses numeric keys "1", "2", etc.
        items = {}
        for i in range(1, 14):
            key = f'pcs_{i:02d}'
            # Try numeric key (new format)
            value = scores_dict.get(str(i))
            items[key] = value
    else:
        # Old format
        scores_dict = result.get('scores', {})
        persona = result.get('persona', {})
        model_reasoning = result.get('model_reasoning')
        
        # Subscales at top level in old format
        pcs_rumination = result.get('rumination')
        pcs_magnification = result.get('magnification')
        pcs_helplessness = result.get('helplessness')
        
        # Individual items - old format uses "pcs_01", "item_1", etc.
        items = {}
        for i in range(1, 14):
            key = f'pcs_{i:02d}'
            value = scores_dict.get(key) or scores_dict.get(f'item_{i}')
            items[key] = value
    
    data = {
        'pcs_total': result.get('total_score'),
        'pcs_rumination': pcs_rumination,
        'pcs_magnification': pcs_magnification,
        'pcs_helplessness': pcs_helplessness,
        'persona_name': persona.get('name'),
        'persona_traits': json.dumps(persona.get('traits')) if persona.get('traits') else None,
        'model_reasoning': model_reasoning
    }
    
    # Add individual items
    data.update(items)
    
    return data

# Extract PCS data
pcs_records = []
for _, row in pcs_raw_df.iterrows():
    pcs_data = parse_pcs_result(row['result_json'])
    pcs_data['experiment_id'] = row['experiment_id']
    pcs_data['experiments_group_id'] = row['experiments_group_id']
    pcs_records.append(pcs_data)

df_llm_pcs = pd.DataFrame(pcs_records)
df_llm_pcs['participant_id'] = df_llm_pcs['experiment_id'].map(exp_to_participant)
df_llm_pcs['model'] = df_llm_pcs['experiment_id'].map(exp_to_model)
df_llm_pcs['run_number'] = RUN_NUMBER

# Filter valid records
df_llm_pcs = df_llm_pcs[df_llm_pcs['participant_id'].notna()].copy()

# Reorder columns
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness',
              'pcs_01', 'pcs_02', 'pcs_03', 'pcs_04', 'pcs_05', 'pcs_06', 'pcs_07',
              'pcs_08', 'pcs_09', 'pcs_10', 'pcs_11', 'pcs_12', 'pcs_13',
              'persona_name', 'persona_traits', 'model_reasoning']
df_llm_pcs = df_llm_pcs[[c for c in cols_order if c in df_llm_pcs.columns]]

print(f"\nExtracted PCS data for {len(df_llm_pcs)} records")
print(f"PCS Total - Mean: {df_llm_pcs['pcs_total'].mean():.2f}, Range: [{df_llm_pcs['pcs_total'].min()}, {df_llm_pcs['pcs_total'].max()}]")

# Diagnostic: Check item extraction
item_cols = [f'pcs_{i:02d}' for i in range(1, 14)]
items_with_data = sum(df_llm_pcs[col].notna().sum() for col in item_cols if col in df_llm_pcs.columns)
total_possible = len(df_llm_pcs) * 13
print(f"Individual items extracted: {items_with_data}/{total_possible} ({100*items_with_data/total_possible:.1f}%)")

## 6. Extract BPI Results

In [ ]:
# Query BPI questionnaire results
query_bpi = f"""
SELECT 
    er.experiment_id,
    er.experiments_group_id,
    er.result_json
FROM {SOURCE_SCHEMA}.evaluation_results er
WHERE er.experiments_group_id = {EXPERIMENTS_GROUP_ID}
  AND er.result_type = 'BPI-IS'
"""

with Session(db.engine) as session:
    bpi_raw_df = pd.read_sql(text(query_bpi), session.connection())

print(f"Loaded {len(bpi_raw_df)} BPI records")

In [ ]:
def parse_bpi_result(result_json):
    """Extract BPI data from result JSON.
    
    Handles two formats:
    - Old format (run 1): responses as array at top level
    - New format (runs 2+): responses inside raw_data as array of objects with code and value
    
    Note: BPI skips question 1_4, so bpiq14 doesn't exist in responses
    """
    if not result_json:
        return {}
    
    result = result_json if isinstance(result_json, dict) else json.loads(result_json)
    
    # BPI skips question 1_4, so bpiq14 doesn't exist in responses  
    item_mapping = {
        0: 'bpiq11', 1: 'bpiq12', 2: 'bpiq13',
        3: 'bpiq15', 4: 'bpiq16', 5: 'bpiq17',
        6: 'bpiq28', 7: 'bpiq39', 8: 'bpiq410', 9: 'bpiq511'
    }
    
    # Check if this is the new format (has raw_data key)
    if 'raw_data' in result:
        raw_data = result['raw_data']
        responses_raw = raw_data.get('responses', [])
        persona = raw_data.get('persona', {})
        model_reasoning = raw_data.get('model_reasoning')
        
        # New format: responses is array of objects with 'code' and 'value'
        # Extract values in order
        responses = [item.get('value') for item in responses_raw]
        
        # Extract subscales from new format
        subscales = result.get('subscales', {})
        bpi_interference_avg = subscales.get('interference_avg')
        bpi_intensity_avg = subscales.get('intensity_avg')
        bpi_interference_total = subscales.get('interference_total')
        bpi_intensity_total = subscales.get('intensity_total')
    else:
        # Old format: responses as array at top level
        responses = result.get('responses', [])
        persona = result.get('persona', {})
        model_reasoning = result.get('model_reasoning')
        
        # Subscales at top level in old format
        bpi_interference_avg = result.get('interference_avg')
        bpi_intensity_avg = result.get('intensity_avg')
        bpi_interference_total = result.get('interference_total')
        bpi_intensity_total = result.get('intensity_total')
    
    bpi_total = result.get('total_score')
    data = {
        'bpi_total': bpi_total,
        # bpi_total_mean: Mean of all items (0-10 scale, matches real data format)
        'bpi_total_mean': bpi_total / 10.0 if bpi_total is not None else None,
        'bpi_interference_avg': bpi_interference_avg,
        'bpi_intensity_avg': bpi_intensity_avg,
        'bpi_interference_total': bpi_interference_total,
        'bpi_intensity_total': bpi_intensity_total,
        'persona_name': persona.get('name'),
        'persona_traits': json.dumps(persona.get('traits')) if persona.get('traits') else None,
        'model_reasoning': model_reasoning
    }
    
    # Individual items
    for idx, item_name in item_mapping.items():
        if idx < len(responses):
            data[item_name] = responses[idx]
    
    return data

# Extract BPI data
bpi_records = []
for _, row in bpi_raw_df.iterrows():
    bpi_data = parse_bpi_result(row['result_json'])
    bpi_data['experiment_id'] = row['experiment_id']
    bpi_data['experiments_group_id'] = row['experiments_group_id']
    bpi_records.append(bpi_data)

df_llm_bpi = pd.DataFrame(bpi_records)
df_llm_bpi['participant_id'] = df_llm_bpi['experiment_id'].map(exp_to_participant)
df_llm_bpi['model'] = df_llm_bpi['experiment_id'].map(exp_to_model)
df_llm_bpi['run_number'] = RUN_NUMBER

# Filter valid records
df_llm_bpi = df_llm_bpi[df_llm_bpi['participant_id'].notna()].copy()

# Reorder columns
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'bpi_total', 'bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg', 'bpi_interference_total', 'bpi_intensity_total',
              'bpiq11', 'bpiq12', 'bpiq13', 'bpiq14', 'bpiq15', 'bpiq16', 'bpiq17',
              'bpiq28', 'bpiq39', 'bpiq410', 'bpiq511',
              'persona_name', 'persona_traits', 'model_reasoning']
df_llm_bpi = df_llm_bpi[[c for c in cols_order if c in df_llm_bpi.columns]]

print(f"\nExtracted BPI data for {len(df_llm_bpi)} records")
print(f"BPI Total Mean (0-10 scale): {df_llm_bpi['bpi_total_mean'].mean():.2f}")
if 'bpi_interference_avg' in df_llm_bpi.columns and df_llm_bpi['bpi_interference_avg'].notna().any():
    print(f"BPI Interference Mean: {df_llm_bpi['bpi_interference_avg'].mean():.2f}")

# Diagnostic: Check item extraction (bpiq14 is always null since BPI skips Q1_4)
bpi_item_cols = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq15', 'bpiq16', 'bpiq17', 'bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
items_with_data = sum(df_llm_bpi[col].notna().sum() for col in bpi_item_cols if col in df_llm_bpi.columns)
total_possible = len(df_llm_bpi) * 10  # Only 10 items, not 11
print(f"Individual items extracted: {items_with_data}/{total_possible} ({100*items_with_data/total_possible:.1f}%)")

## 7. Extract TSK Results

In [ ]:
# Query TSK questionnaire results
query_tsk = f"""
SELECT 
    er.experiment_id,
    er.experiments_group_id,
    er.result_json
FROM {SOURCE_SCHEMA}.evaluation_results er
WHERE er.experiments_group_id = {EXPERIMENTS_GROUP_ID}
  AND er.result_type = 'TSK-11SV'
"""

with Session(db.engine) as session:
    tsk_raw_df = pd.read_sql(text(query_tsk), session.connection())

print(f"Loaded {len(tsk_raw_df)} TSK records")

In [ ]:
def parse_tsk_result(result_json):
    """Extract TSK data from result JSON.
    
    Handles two formats:
    - Old format (run 1): responses as array at top level
    - New format (runs 2+): responses inside raw_data as array of objects with code and value
    """
    if not result_json:
        return {}
    
    result = result_json if isinstance(result_json, dict) else json.loads(result_json)
    
    # Check if this is the new format (has raw_data key)
    if 'raw_data' in result:
        raw_data = result['raw_data']
        responses_raw = raw_data.get('responses', [])
        persona = raw_data.get('persona', {})
        model_reasoning = raw_data.get('model_reasoning')
        
        # New format: responses is array of objects with 'code' and 'value'
        # Extract values in order
        responses = [item.get('value') for item in responses_raw]
    else:
        # Old format: responses as array at top level
        responses = result.get('responses', [])
        persona = result.get('persona', {})
        model_reasoning = result.get('model_reasoning')
    
    data = {
        'tsk_total': result.get('total_score'),
        'persona_name': persona.get('name'),
        'persona_traits': json.dumps(persona.get('traits')) if persona.get('traits') else None,
        'model_reasoning': model_reasoning
    }
    
    # Individual items (stored as strings per schema)
    for i in range(1, 12):
        key = f'tsk_{i:02d}'
        if i-1 < len(responses):
            data[key] = str(responses[i-1]) if responses[i-1] is not None else None
    
    return data

# Extract TSK data
tsk_records = []
for _, row in tsk_raw_df.iterrows():
    tsk_data = parse_tsk_result(row['result_json'])
    tsk_data['experiment_id'] = row['experiment_id']
    tsk_data['experiments_group_id'] = row['experiments_group_id']
    tsk_records.append(tsk_data)

df_llm_tsk = pd.DataFrame(tsk_records)
df_llm_tsk['participant_id'] = df_llm_tsk['experiment_id'].map(exp_to_participant)
df_llm_tsk['model'] = df_llm_tsk['experiment_id'].map(exp_to_model)
df_llm_tsk['run_number'] = RUN_NUMBER

# Filter valid records
df_llm_tsk = df_llm_tsk[df_llm_tsk['participant_id'].notna()].copy()

# Reorder columns
item_cols = [f'tsk_{i:02d}' for i in range(1, 12)]
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'tsk_total'] + item_cols + ['persona_name', 'persona_traits', 'model_reasoning']
df_llm_tsk = df_llm_tsk[[c for c in cols_order if c in df_llm_tsk.columns]]

print(f"\nExtracted TSK data for {len(df_llm_tsk)} records")
if 'tsk_total' in df_llm_tsk.columns and df_llm_tsk['tsk_total'].notna().any():
    print(f"TSK Total - Mean: {df_llm_tsk['tsk_total'].mean():.2f}, Range: [{df_llm_tsk['tsk_total'].min()}, {df_llm_tsk['tsk_total'].max()}]")

# Diagnostic: Check item extraction
items_with_data = sum(df_llm_tsk[col].notna().sum() for col in item_cols if col in df_llm_tsk.columns)
total_possible = len(df_llm_tsk) * 11
print(f"Individual items extracted: {items_with_data}/{total_possible} ({100*items_with_data/total_possible:.1f}%)")

## 8. Summary Before Save

In [ ]:
print("="*60)
print(f"EXTRACTION SUMMARY - Run {RUN_NUMBER}")
print("="*60)
print(f"\nExperiment Group: {EXPERIMENTS_GROUP_ID}")
print(f"Run Number: {RUN_NUMBER}")
print(f"\nRecords to insert:")
print(f"  llm_dimension_evaluation: {len(df_llm_dimensions)}")
print(f"  llm_pcs_results: {len(df_llm_pcs)}")
print(f"  llm_bpi_results: {len(df_llm_bpi)}")
print(f"  llm_tsk_results: {len(df_llm_tsk)}")

all_152 = (len(df_llm_dimensions) == 152 and len(df_llm_pcs) == 152 and 
           len(df_llm_bpi) == 152 and len(df_llm_tsk) == 152)
print(f"\nAll tables have 152 records: {'YES' if all_152 else 'NO - CHECK BEFORE SAVING'}")

## 9. Save to ACM Schema

In [ ]:
# Save dimension evaluations
print('Saving llm_dimension_evaluation...')
df_llm_dimensions.to_sql(
    'llm_dimension_evaluation',
    db.engine,
    schema=TARGET_SCHEMA,
    if_exists='append',
    index=False
)
print(f'  Saved {len(df_llm_dimensions)} rows')

In [ ]:
# Save PCS results
print('Saving llm_pcs_results...')
df_llm_pcs.to_sql(
    'llm_pcs_results',
    db.engine,
    schema=TARGET_SCHEMA,
    if_exists='append',
    index=False
)
print(f'  Saved {len(df_llm_pcs)} rows')

In [ ]:
# Save BPI results
print('Saving llm_bpi_results...')
df_llm_bpi.to_sql(
    'llm_bpi_results',
    db.engine,
    schema=TARGET_SCHEMA,
    if_exists='append',
    index=False
)
print(f'  Saved {len(df_llm_bpi)} rows')

In [ ]:
# Save TSK results
print('Saving llm_tsk_results...')
df_llm_tsk.to_sql(
    'llm_tsk_results',
    db.engine,
    schema=TARGET_SCHEMA,
    if_exists='append',
    index=False
)
print(f'  Saved {len(df_llm_tsk)} rows')

## 10. Verification

In [ ]:
# Verify records by run_number
print("\n" + "="*60)
print("VERIFICATION - Records by run_number")
print("="*60)

llm_tables = ['llm_dimension_evaluation', 'llm_pcs_results', 'llm_bpi_results', 'llm_tsk_results']

with Session(db.engine) as session:
    for table in llm_tables:
        query = f"""
        SELECT run_number, COUNT(*) as count
        FROM {TARGET_SCHEMA}.{table}
        GROUP BY run_number
        ORDER BY run_number
        """
        counts = pd.read_sql(text(query), session.connection())
        print(f"\n{table}:")
        for _, row in counts.iterrows():
            status = 'OK' if row['count'] == 152 else 'CHECK!'
            print(f"  Run {row['run_number']}: {row['count']} records [{status}]")

In [ ]:
print("\n" + "="*60)
print(f"EXTRACTION COMPLETE - Run {RUN_NUMBER}")
print("="*60)